In [10]:
# TODO make sure this renders in the github repo

# Configurations For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)


- **Layers:** Noted as **num_layers**. How many Decoder layers the model has.
- **Model Dimensions:** Also noted as **d_model** or $d_{model}$. It represents the expected features for each token in the input and output sequences.
    - Example: when the Token Embeddings layer happen, each token is converted into a dense vector of size $d_{model}$.
- **FFN Dimensions:** Noted as **d_ff**. Is the hidden size of the Feed Forward SwiGLU sub-layer. We use this to expand the linear layers for SwiGLU, which gives the model more parameters to learn non-linear functions.
- **Attention Heads:** This the the total number of Query heads, note K & V use the Key/Value Heads hyperparameter.
    - If d_model = $4{,}096$ and attn_heads = $32$, each individual head processes a vector of size $4{,}096/32 = 128$
- **Key/Value Heads:** Noted as **num_kv_heads**. The number of key and value heads in the attention mechanism. The reason the number of Key and Value heads is smaller than the number of Query heads, is because this allows faster generation, larger batch sizes, and reduces the memory footprint.
- **Peak Learning Rate:** At the end of the warmup, the learning rate hits the peak it can be, then it starts to decay. This keeps training stable, and prevents exploding gradients early on when the weights are random.

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length (context_len/sequence_len):** The model's short-term memory. Represents how much words/tokens the model can remember at one time. So, as you chat with the model, it loses track of earlier context that no longer fits its context window. In the data, it is the number of tokens in a single document/row in a list of documents.

- **Positional Embeddings** The frequency that the Q and K tokens are rotated by, to give them positional information.


The [Llama-3.1 8B tokenizer](https://huggingface.co/meta-llama/Llama-3.1-8B/blob/main/tokenizer_config.json) consists of $128{,}255$ tokens.

- **Standard tokens:** IDs $0$ through $127{,}999$ ($128{,}000$ total) are the standard tokens.
- **Special tokens:** IDs $128{,}000$ through $128{,}255$ ($256$ total) are the special tokens.
- **Llama 3 paper:** "We use a vocabulary with 128K tokens. Our token vocabulary combines 100K tokens from the tiktoken tokenizer with 28K additional tokens to better support non-English languages. Compared to the Llama 2 tokenizer, our new tokenizer improves compression rates on a sample of English data from 3.17 to 3.94 characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding 28K tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."


**Tree structure:**

```text
.
├── artifacts
│   ├── checkpoints
│   │   ├── Scaled_down_Llama_3_1_Base
│   │   │   ├── config.json
│   │   │   ├── latest.txt # contains"step_000250.pt"
│   │   │   ├── step_000250_training_loss_plot.png
│   │   │   └── step_000250.pt
│   │   ├── Scaled_down_Llama_3_1_SFT
│   │   │   ├── config.json
│   │   │   ├── latest.txt
│   │   │   └── step_002000.pt
│   │   └── Scaled_down_Llama_3_1_Instruct (Inference models)
│   │       ├── config.json
│   │       ├── latest.txt
│   │       └── step_001000.pt
│   ├── meta-models # Meta's trained models # TODO will I save them locally or just load in memory?
│   ├── models # My trained models
│   └── universal_tokenizer # For my models
│       ├── bpe_vocab_size_32000_samples_1000_info.json
│       └── bpe_vocab_size_32000_samples_1000.json
└── data
    ├── initial_and_lc_stage.bin
    ├── initial_and_lc_stage.json
    ├── initial_and_lc_stage.parquet
    .
    .
```


In [11]:
# TODO At end of project, make sure tree above is correct!


In [12]:
# TODO add below markdown back after adding gradient accumulation back to the project

**Gradient Accumulation**: 
- A technique that allows us to perform multiple forward and backward passes, but instead of updating the weights immediately, we **accumulate** the gradients over **several steps**. Only after the specified number of `accumulation steps` do we trigger the optimizer to update the weights. This means we can simulate a massive Global Batch Size that would otherwise not fit in our hardware's VRAM.

- **Global Batch Size**: The total number of tokens the optimizer needs to look at together to calculate an accurate gradient. $\text{Global Batch Size} = \text{Micro Batch Size} \times \text{Number of GPUs} \times \text{Accumulation Steps}$
- **Micro Batch Size:** The number of sequences your hardware can hold in its VRAM (GPU Memory) during a single forward and backward pass.
    - Example: `micro_batch_size` $=4$ and `context_len` $=4{,}096$ a single batch will contain $4*4{,}096 = 16{,}384 \, \text{tokens}$ per GPU.
- We also scale the the accumulated gradients. $\text{Scaled Loss} = \text{Loss} / \text{Accumulation Steps}$

# Configurations:


In [13]:
import easyjupyter

import os
from pathlib import Path
import torch
from utils.fs_manager import find_root, create_structure
from utils.misc import  print_config, get_device
from pydantic import BaseModel, Field, model_validator, PrivateAttr
from typing import Optional, ClassVar, Any
import numpy as np

## Configurations Template

In [ ]:
class DatasetConfig(BaseModel):
    name: str
    config: str


class InitialStageConfig(BaseModel):
    """Initial stage of the text-only pre-training."""

    def get_batch_size(self, tokens_processed: int) -> tuple[int, int]:
        """
        The batch size and context length is doubled after a certain number of steps in the initial stage.
             Paper: "Specifically, we use an initial batch size of 4M tokens and sequences of length 4,096, and double these values to a batch size of 8M sequences of 8,192 tokens after pre-training 252M tokens. We double the batch size again to 16M after pre-training on 2.87T tokens."

        Args:
            tokens_processed (int): The number of tokens already processed prior to the current step.
        Returns:
            tuple[int, int]: (batch_size, context_len)
        """
        # TODO: when i redo pretrain maybe theres a better way to handle the switches
        if tokens_processed < self.first_batch_and_context_increase_at:
            # Paper: "we use an initial batch size of 4M tokens and sequences of length 4,096"
            return self.initial_batch_size, self.initial_content_len
        elif tokens_processed <= self.first_batch_and_context_increase_at:
            # Paper: "double these values to a batch size of 8M sequences of 8,192 tokens after pre-training 252M tokens"
            new_batch_size = self.initial_batch_size * 2
            new_context_len = self.initial_content_len * 2
            return new_batch_size, new_context_len
        elif tokens_processed >= self.second_batch_and_context_increase_at:
            # Paper: "We double the batch size again to 16M after pre-training on 2.87T tokens."
            new_batch_size = self.initial_batch_size * 4
            context_len = self.initial_content_len * 2
            return new_batch_size, context_len

    tokens: int  # The number of tokens to process for this phase.
    warmup_steps: int
    initial_content_len: int
    initial_batch_size: int

    tokenizer_training_samples: int  # How many samples/documents to stream from HuggingFace dataset to train a tokenizer with.

    first_batch_and_context_increase_at: int
    second_batch_and_context_increase_at: int

    gradient_accumulation_steps: Optional[int] = 10  # TODO is needed?
    dataset: DatasetConfig = Field(
        default_factory=lambda: DatasetConfig(
            name="HuggingFaceFW/fineweb-edu",
            config="sample-10BT",
        )
    )
    peak_lr: float
    decay_lr_to: float


class Lc_StageConfig(BaseModel):
    """Long-Context stage of the text-only pre-training."""

    tokens: int
    content_len: int
    increase_context_len_to: int  # Paper: "we increased context length gradually in six stages, starting from the original 8K context window and ending in the final 128K context window"
    gradient_accumulation_steps: Optional[int] = 10  # TODO is needed?
    dataset: DatasetConfig = Field(
        default_factory=lambda: DatasetConfig(
            name="HuggingFaceFW/fineweb-edu",
            config="sample-10BT",
        )
    )


class AnnealingStageConfig(BaseModel):
    """Annealing stage of the text-only pre-training."""

    tokens: int
    decay_lr_to: float
    content_len: int
    gradient_accumulation_steps: Optional[int] = 10  # TODO is needed?
    dataset: DatasetConfig = Field(
        default_factory=lambda: DatasetConfig(
            name="HuggingFaceFW/finepdfs",
            config="eng_Latn",
        )
    )


class PretrainConfig(BaseModel):
    training_steps: int

    initial_stage: InitialStageConfig
    lc_stage: Lc_StageConfig
    annealing_stage: AnnealingStageConfig


class TextOnlyConfig(BaseModel):
    pretrain: PretrainConfig
    train_sft: Optional[dict] = None  # TODO
    train_dpo: Optional[dict] = None  # TODO


class ConfigTemplate(BaseModel):

    _initialized: bool = PrivateAttr(default=False)

    # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    num_workers: int = 0

    model_name: str
    rope_theta: int
    rms_norm_eps: float
    d_model: int
    num_layers: int
    d_ff: int
    attn_heads: int
    num_kv_heads: int
    vocab_size: int

    chunk_size: int  # The number of documents that we tokenize at a time in parallel across all CPU cores. This is used when building the binary dataset from raw-text dataset using the tokenizer. The tokenizer has its own chunk_size.

    # Date type for storing the binary dataset. If custom BPE tokenizer vocab is < 65K, use np.uint16 to save 50% of space. The Llama3 vocab was 128K so it must use np.uint32.
    dtype: Any = np.uint16

    text_only: TextOnlyConfig
    multimodal: Optional[dict] = None  # TODO

    step_counter: int = 0

    # This increments, and will not be saved to JSON file, the context_len in the stages will be saved to the JSON file.
    context_len: int = Field(default=0, exclude=True)

    # Managing fs resources, excluded form serialization
    _project_root: Path = PrivateAttr(default_factory=find_root)
    _data_dir: Path = PrivateAttr()
    _artifacts_dir: Path = PrivateAttr()
    _chpts_dir: Path = PrivateAttr()
    _curr_chpt_dir: Optional[Path] = PrivateAttr(default=None)
    _device: torch.device = PrivateAttr(default_factory=get_device)

    special_tokens: ClassVar[dict[str, dict[str, str | int]]] = {
        "pad_token": {"token": "<|pad_token|>", "ID": 0},
        # The token for the beginning of the text.
        "doc_begin_token": {"token": "<|begin_of_doc|>", "ID": 1},
        # The token for the end of document, it is injected between documents.
        "doc_end_token": {"token": "<|end_of_doc|>", "ID": 2},
        # unknown token.
        "unk_token": {"token": "<|unk|>", "ID": 3},
    }

    @model_validator(mode="after")
    def setup_execution_state(self) -> ConfigTemplate:
        """Automatically called by Pydantic after validation to set up flat state variables. Setups up the file system structure and system variables."""
        # Skip initialization at definition, you need to use .initialize() to explicitly initialize the config in main.py
        return self

    def initialize(self, is_overfit=False):
        # Had to implement this to avoid it initializing the config on definition.
        self._data_dir, self._artifacts_dir, self._chpts_dir = create_structure(
            self._project_root
        )
        print(f"Using config: {self.model_name}")

        if is_overfit:
            print("Configured for overfitting...")

            # TODO Set all the variables for overfit
            self.text_only.pretrain.initial_stage.tokenizer_training_samples = 100
            self.text_only.pretrain.initial_stage.tokens = 10_000
            self.text_only.pretrain.lc_stage.tokens = 10_000
            self.text_only.pretrain.annealing_stage.tokens = 10_000
            self.chunk_size = 1000

        return self

    @property
    def project_root(self) -> Path:
        return self._project_root

    @property
    def data_dir(self) -> Path:
        return self._data_dir

    @property
    def artifacts_dir(self) -> Path:
        return self._artifacts_dir

    @property
    def chpts_dir(self) -> Path:
        return self._chpts_dir

    @property
    def device(self) -> torch.device:
        return self._device

    @property
    def curr_chpt_dir(self) -> Optional[Path]:
        return self._curr_chpt_dir

    @curr_chpt_dir.setter
    def curr_chpt_dir(self, path: Path):
        self._curr_chpt_dir = path

    def save(self, filepath: Path):
        """Native Pydantic serialization"""
        filepath.parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, "w") as f:
            f.write(self.model_dump_json(indent=4))
        print(f"Saved Config to -> {filepath}")

    @classmethod
    def load(cls, filepath: Path):
        """Native Pydantic deserialization"""
        with open(filepath, "r") as f:
            instance = cls.model_validate_json(f.read())
        print(f"Loaded Config from -> {filepath}")
        return instance

    def __str__(self) -> str:
        return print_config(self)

    def __repr__(self) -> str:
        return print_config(self)

## Llama 3.1 405B Configuration

💡 The 405B model is the largest model from the Llama 3 family, with 405 Billion parameters, it is a text-only model, and is also the most well documented in the Llama 3 paper. I will not be training this model, as it is too large. Instead, I will create the configuration for the 405B model, build out the training pipeline, and then create a scaled down configuration version of it, and train that instead.

**Llama 3 paper:** "**Language model pre-training.** We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train **a model with 405B parameters** on **15.6T tokens using a context window of 8K tokens**. This standard pre-training stage is followed by a continued pre-training stage that increases the supported **context window to 128K tokens**."


---

**PAPER:**

**3.4.1 Initial Pre-Training**

We pre-train Llama $3$ $405\text{B}$ using AdamW with a peak learning rate of $8 × 10^{−5}$ , a linear warm up of $8{,}000$ steps, and a cosine learning rate schedule decaying to $8 × 10^{−7}$ over $1{,}200{,}000$ steps. We use a lower batch size early in training to improve training stability, and increase it subsequently to improve efficiency. Specifically, we use an initial batch size of $4\text{M}$ tokens and sequences of length $4{,}096$, and double these values to a batch size of $8\text{M}$ sequences of $8{,}192$ tokens after pre-training $252\text{M}$ tokens. We double the batch size again to 16M after pre-training on $2.87\text{T}$ tokens. We found this training recipe to be very stable: we observed few loss spikes and did not require interventions to correct for model training divergence.

**3.4.2 Long Context Pre-Training**

In the final stages of pre-training, we train on long sequences to support context windows of up to $128\text{K}$ tokens. We do not train on long sequences earlier because the compute in self-attention layers grows quadratically in the sequence length. We increase the supported context length in increments, pre-training until the model has successfully adapted to the increased context length. We assess successful adaptation by measuring whether (1) model performance on short-context evaluations has recovered completely and (2) the model perfectly solves “needle in a haystack” tasks up to that length. In Llama $3$ $405\text{B}$ pre-training, we increased context length gradually in six stages, starting from the original $8\text{K}$ context window and ending in the final $128\text{K}$ context window. This long-context pre-training stage was performed using approximately $800\text{B}$ training tokens.

**3.4.3 Annealing**

During pre-training on the final $40\text{M}$ tokens, we linearly annealed the learning rate to $0$, maintaining a context length of $128\text{K}$ tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model.

---

🚨 **Notes:**

- I will not be using the same datasets that Meta.
- For attributes like the value for the RMSNorm `rms_norm_eps`, which are not mentioned in the paper, I referenced from the official [HuggingFace Meta 405B config.json](https://huggingface.co/meta-llama/Llama-3.1-405B/blob/main/config.json)

In [15]:
Llama3_1_405B = ConfigTemplate(
    # Llama 3.1 405B Configuration. This model is not multi-modal, it only takes in text!
    model_name="Llama 3.1 405B",
    rope_theta=500_000,
    rms_norm_eps=1e-5,
    d_model=16_384,
    num_layers=126,
    dtype=np.uint32,
    d_ff=53_248,
    attn_heads=128,
    num_kv_heads=8,
    vocab_size=128_000,
    chunk_size=64_000, # TODO a good number for the 405B model?
    text_only=TextOnlyConfig(
        pretrain=PretrainConfig(
            training_steps=1_200_000,  # TODO in 1,200,000 steps only for the initial stage?
            initial_stage=InitialStageConfig(
                tokens=15_600_000_000_000,
                tokenizer_training_samples=50_000,
                initial_batch_size=4_000_000,
                initial_content_len=4_096,
                first_batch_and_context_increase_at=252_000_000,
                second_batch_and_context_increase_at=2_870_000_000_000,
                warmup_steps=8_000,
                peak_lr=8e-5,
                decay_lr_to=8e-7,
            ),
            lc_stage=Lc_StageConfig(
                tokens=800_000_000_000,
                content_len=8_192,
                increase_context_len_to=128_000,
            ),
            annealing_stage=AnnealingStageConfig(
                content_len=128_000, tokens=40_000_000, decay_lr_to=0.0
            ),
        )
    ),
)

In [21]:
# @i-c
Llama3_1_405B.initialize()
print(Llama3_1_405B)


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: Llama 3.1 405B


==================== Config ====================
            [Model Architecture]
                - model_name               = Llama 3.1 405B
                - d_model                  = 16384
                - num_kv_heads             = 8
                - d_ff                     = 53248
                - num_layers               = 126
                - attn_heads               = 128
                - rms_norm_eps             = 1e-05
                - rope_theta               = 500000
                - vocab_size               = 128000
            

            [Active Run]
                - Runtime [context_len]    = 0
                - num_workers              = 0
                - device                   = mps
                - step_counter             = 0
            

            [Stages Configs]
                [Pre-Training Stage]
                    - token_budget         = None
                 

## Llama 3.1 8B Configuration


## Scaled Down Llama 3.1 Architecture

- Scaled down version of the Llama 3.1 architecture.


In [18]:
# TODO: copy and paste form the 405B model and scale down, do not use percentages!

# import numpy as np

# class Llama3_scaled_down(BaseConfig):
#     """
#     Scaled down version of the Llama 3.1 architecture.
#     """

#     model_name = "Scaled down Llama 3.1"

#     num_layers = 8
#     d_model = 768
#     d_ff = 2048
#     num_kv_heads = 4
#     attn_heads = 12
#     rms_norm_eps = 1e-5
#     vocab_size = 32_000

#     # The context length will increase to from the initial length to the max, during the long-context stage of pre-training in 6 training steps.
#     initial_context_len = 256
#     context_len = 256
#     max_context_len = 1024

#     micro_batch_size = 8
#     gradient_accumulation_steps = 16
#     # TODO learning rate is different in SFT than pre-training
#     peak_lr = 5e-4  # Higher peak learning rate for smaller models
#     warmup_steps = 500
#     pos_embeddings_freq = 10_000


#     # ================== Training ==================
#     step_counter = 0 # Saved inside the .pt state dict, and not in the config.json

#     # Date type for storing the binary dataset.
#     # If custom BPE tokenizer vocab is < 65K, use np.uint16 to save 50% of space. The Llama3 vocab was 128K so it must use np.uint32.
#     dtype = np.uint16

#     def __init__(self):
#         super().__init__()

#         self.pre_training_stages_steps = {
#             # More info in pre_training.ipynb
#             "initial_perc": 0.90,  # Initial stage will last 90% of the total training steps
#             "long_context_perc": 0.07,  # Long-Context stage lasts 7% of the total training steps
#             "annealing_perc": 0.03,  # Annealing stage runs from 3%
#         }

## Llama 3.2 11B Configuration.

This config is for **multi-modal for vision and text**!
